# Cogu — treinar a política de caminhada em GPU (Colab)

Treina a política do robô em uma GPU NVIDIA do Colab, com o mesmo código do repositório
(`sim/playground/train.py`), e devolve um checkpoint para assistir no Mac com o `play.py`.

1. **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)**.
2. **Ambiente de execução → Executar tudo**.
3. No fim, baixe o `.zip` e descompacte dentro de `build/checkpoints/` no projeto local:

```bash
unzip ~/Downloads/cogu_walk_gpu.zip -d build/checkpoints/
uv run mjpython sim/playground/play.py --checkpoint build/checkpoints/cogu_walk_gpu
```

O MJCF é gerado a partir do CAD aqui mesmo (`cad.export`), como no CI: nada é editado à mão.

In [ ]:
# 1. GPU disponível?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Código e dependências (uv + JAX com CUDA 12, mesma versão do projeto)
import os
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
REPO = "https://github.com/laroccathebrux/cogu.git"
BRANCH = "main"
if not os.path.exists("/content/cogu"):
    !git clone --depth 1 -b {BRANCH} {REPO} /content/cogu
%cd /content/cogu
!git log --oneline -1
!uv sync --frozen -q
# o lockfile tem o JAX de CPU (o projeto roda no Mac); aqui somamos o plugin CUDA da mesma versão.
# O Colab define UV_SYSTEM_PYTHON=true: sem --python o uv instalaria no Python do sistema.
# Daqui em diante tudo roda com `uv run --no-sync`, para o uv não desfazer essa instalação.
!uv pip install -q --python .venv/bin/python "jax[cuda12]==0.11.2"
!uv run --no-sync python -c "import jax; print('JAX', jax.__version__, jax.devices())"

In [ ]:
# 3. Gerar o modelo a partir do CAD e conferir que o robô fica em pé
!uv run --no-sync python -m cad.export --out build --no-step | tail -4
!uv run --no-sync python sim/check_mjcf.py build/mjcf/cogu_lower_body.xml | tail -2
!uv run --no-sync python sim/playground/check_env.py 2>&1 | grep --line-buffered -v Warning | tail -2

In [ ]:
# 4. Configuração do treino
NAME = "cogu_walk_gpu"     #@param {type:"string"}
STEPS = 50_000_000         #@param {type:"integer"}
NUM_ENVS = 4096            #@param {type:"integer"}
BATCH_SIZE = 512           #@param {type:"integer"}
NUM_MINIBATCHES = 8        #@param {type:"integer"}
NUM_EVALS = 10             #@param {type:"integer"}
SEED = 0                   #@param {type:"integer"}
assert BATCH_SIZE * NUM_MINIBATCHES % NUM_ENVS == 0, "batch_size × num_minibatches precisa ser múltiplo de num_envs"
OUT = f"build/checkpoints/{NAME}"

In [ ]:
# 5. Treino (uma linha por avaliação: recompensa, duração do episódio, velocidade média)
!uv run --no-sync python -u sim/playground/train.py --steps {STEPS} --num-envs {NUM_ENVS} \
    --batch-size {BATCH_SIZE} --num-minibatches {NUM_MINIBATCHES} --num-evals {NUM_EVALS} \
    --seed {SEED} --out {OUT} 2>&1 | grep --line-buffered -v -i warning

In [ ]:
# 6. Sim2sim: a política no MuJoCo normal (CPU), sem e com ruído/atraso
for extra in ["--vx 0", "--vx 0.15", "--vx 0.25", "--vx 0.15 --noise 1 --delay 1", "--vx 0.1 --wz 0.8"]:
    print("==", extra)
    !uv run --no-sync python sim/playground/play.py --headless --seconds 8 --checkpoint {OUT} {extra} 2>&1 | grep --line-buffered -v -i warning | tail -1

In [ ]:
# 7. Vídeo de 8 s a 0,15 m/s
os.environ["MUJOCO_GL"] = "egl"
!uv run --no-sync python sim/playground/rollout.py --checkpoint {OUT} --seconds 8 --out build/videos/{NAME}.mp4 2>&1 | grep --line-buffered -v -i warning | tail -3
from IPython.display import Video
Video(f"build/videos/{NAME}.mp4", embed=True, width=640)

In [ ]:
# 8. Empacotar: último checkpoint + progress.jsonl + vídeo
import pathlib, shutil, zipfile
out = pathlib.Path(OUT)
steps = sorted(p for p in out.iterdir() if p.is_dir() and p.name.isdigit())
last = steps[-1]
zip_path = pathlib.Path(f"/content/{NAME}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in last.rglob("*"):
        z.write(f, f"{NAME}/{last.name}/{f.relative_to(last)}")
    z.write(out / "progress.jsonl", f"{NAME}/progress.jsonl")
    if (out / "env_config.json").exists():   # play.py / rollout.py rebuild the action filter from it
        z.write(out / "env_config.json", f"{NAME}/env_config.json")
    video = pathlib.Path(f"build/videos/{NAME}.mp4")
    if video.exists():
        z.write(video, f"{NAME}/{NAME}.mp4")
print(zip_path, f"{zip_path.stat().st_size/1e6:.1f} MB", "checkpoint", last.name)

In [ ]:
# 9. Baixar o zip para o computador
from google.colab import files
files.download(str(zip_path))